# Detection Box Drawing Interface

An interactive Jupyter notebook for drawing bounding boxes around bird songs on
dual-channel spectrograms. Box coordinates (time and frequency) are recorded for
each [HawkEars](https://github.com/jhuus/HawkEars) detection and exported to CSV
for downstream amplitude extraction (e.g., RMS dBFS via SoX).

This tool was developed for the limited-amplitude distance-truncation framework
described in Lebeuf-Taylor et al. (2025).

## 1. Setup and Configuration

Edit the cell below to match your project. All user-configurable settings are here.

In [ ]:
import pandas as pd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import PowerNorm
from matplotlib.widgets import RectangleSelector
import ipywidgets as widgets
from IPython.display import display, clear_output
import sounddevice as sd
from pathlib import Path
from datetime import datetime
import warnings

from utils import (
    parse_audio_filename,
    build_audio_file_lookup,
    load_audio_stereo,
    play_audio,
)

%matplotlib widget
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# USER CONFIGURATION - Edit these settings for your project
# =============================================================================

# Path to your HawkEars detection CSV (already filtered to detections of interest).
# Expected columns: filename, species, confidence, start_time, end_time,
#                    location, recording_datetime
DETECTIONS_CSV = "path/to/your/filtered_detections.csv"

# Directories containing your audio (.wav) files.
# The notebook will recursively search these for .wav files.
AUDIO_DIRS = [
    r"path/to/audio/directory_1",
    r"path/to/audio/directory_2",
]

# Species codes to process. Set to None to auto-detect all species in the CSV.
FOCAL_SPECIES = None  # e.g. ['OVEN', 'BRCR'] or None for all species

# Output directory for results (created automatically)
OUTPUT_DIR = "detection_boxes_output"

# =============================================================================
# OPTIONAL - Filename parsing
# =============================================================================
# HawkEars output filenames typically follow: LocationID_YYYYMMDD_HHMMSS_HawkEars.txt
# The audio .wav files follow:               LocationID_YYYYMMDD_HHMMSS.wav
#
# If your filenames use a different delimiter or structure, edit
# parse_audio_filename() in utils.py.

# Characters in filenames to replace with '_' before parsing
FILENAME_SANITIZE_CHARS = ['$']

# Prefixes to strip from filenames before parsing (e.g. channel prefixes)
FILENAME_STRIP_PREFIXES = ['0+1_']

# =============================================================================
# OPTIONAL - Spectrogram display defaults
# =============================================================================
DEFAULT_DYNAMIC_RANGE = 65  # dB - lower = more contrast
DEFAULT_GAMMA = 1.0         # Power normalization gamma

In [ ]:
# Create output directory
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Derived file paths
PROGRESS_FILE = str(Path(OUTPUT_DIR) / "detection_boxes_progress.csv")
RESULTS_FILE = str(Path(OUTPUT_DIR) / "detection_boxes_results.csv")
FILE_LOOKUP_FILE = str(Path(OUTPUT_DIR) / "audio_file_lookup.csv")

print(f"Output directory: {OUTPUT_DIR}")
print(f"Progress file:   {PROGRESS_FILE}")
print(f"Results file:    {RESULTS_FILE}")
print(f"File lookup:     {FILE_LOOKUP_FILE}")

## 2. Build Audio File Lookup

Searches your audio directories and builds a lookup table mapping HawkEars detection
filenames to audio file paths on disk. Results are cached after the first run.

In [ ]:
# Build or load lookup table
audio_lookup = build_audio_file_lookup(
    AUDIO_DIRS, FILE_LOOKUP_FILE,
    sanitize_chars=FILENAME_SANITIZE_CHARS,
    strip_prefixes=FILENAME_STRIP_PREFIXES
)
print(f"\nAudio lookup: {len(audio_lookup)} files")
audio_lookup.head()

## 3. Load and Prepare Detections

Loads the detection CSV and merges with the audio file lookup so each detection
has a path to its source audio file.

In [ ]:
# Load detections
detections = pd.read_csv(DETECTIONS_CSV)

# Clean column names (remove BOM if present)
detections.columns = detections.columns.str.replace('\ufeff', '')

# Derive filename_base from the HawkEars output filename column
detections['filename_base'] = detections['filename'].str.replace(
    '_HawkEars.txt', '', regex=False
)

# Auto-detect species if not specified
if FOCAL_SPECIES is None:
    FOCAL_SPECIES = sorted(detections['species'].unique().tolist())
    print(f"Auto-detected {len(FOCAL_SPECIES)} species: {FOCAL_SPECIES}")
else:
    print(f"Focal species: {FOCAL_SPECIES}")

# Filter to focal species
detections = detections[detections['species'].isin(FOCAL_SPECIES)].copy()
print(f"Detections for focal species: {len(detections)}")

# Merge with audio file lookup
detections = detections.merge(
    audio_lookup[['location', 'recording_datetime', 'filename_base', 'file_path']],
    on=['location', 'recording_datetime', 'filename_base'],
    how='left'
)

n_missing = detections['file_path'].isna().sum()
print(f"Detections with missing audio files: {n_missing}")

# Remove rows with missing audio files
detections = detections[detections['file_path'].notna()].copy()

# Add unique ID for tracking
detections['detection_id'] = range(len(detections))

print(f"Detections ready for box drawing: {len(detections)}")
print(f"\nDetections per species:")
print(detections['species'].value_counts())

## 4. Run Detection Box Drawing Interface

Execute the cell below to start the interactive box drawing session.

**How to use:**
1. Draw a bounding box around the bird song by clicking and dragging on either spectrogram
2. The box is mirrored on both channels automatically
3. Click a status button to record the detection and advance:

| Button | Meaning |
|--------|---------|
| **Complete** | Box drawn successfully |
| **Not Detected** | Song not visible on spectrogram |
| **Skip** | Revisit later |
| **Bad Weather** | Spectrogram obscured by weather noise |
| **Malfunction** | Recording malfunction |

**Tips:**
- Use the dB Range and Gamma sliders to adjust spectrogram contrast
- Use the mic exclusion checkboxes to flag problematic channels
- Progress auto-saves every 20 detections
- Use the Back button to revisit the previous detection

In [ ]:
class AmplitudeExtractionInterface:
    """
    GUI for drawing bounding boxes around bird songs in dual-channel spectrograms.
    Records box coordinates (time, frequency) for downstream amplitude extraction.
    """

    def __init__(self, detections_df, progress_file=PROGRESS_FILE,
                 results_file=RESULTS_FILE):
        self.detections = detections_df.copy()
        self.progress_file = progress_file
        self.results_file = results_file
        self.current_idx = 0
        self.results = []
        self.total_detections = len(detections_df)

        # Current bounding box coordinates
        self.current_box = None  # (time_start, time_end, freq_min, freq_max)

        # Current audio cache
        self.current_audio_left = None
        self.current_audio_right = None
        self.current_sr = None

        # Spectrogram settings
        self.dynamic_range = DEFAULT_DYNAMIC_RANGE
        self.gamma = DEFAULT_GAMMA

        # Figure and axes references
        self.fig = None
        self.ax_left = None
        self.ax_right = None
        self.rect_selector_left = None
        self.rect_selector_right = None
        self.box_patch_left = None
        self.box_patch_right = None

        self._load_progress()
        self._create_widgets()

    def _load_progress(self):
        """Load previously processed detections."""
        if Path(self.progress_file).exists():
            progress_df = pd.read_csv(self.progress_file)
            self.results = progress_df.to_dict('records')

            processed_ids = set(progress_df['detection_id'])
            all_ids = set(self.detections['detection_id'])
            remaining_ids = all_ids - processed_ids

            self.detections = self.detections[
                self.detections['detection_id'].isin(remaining_ids)
            ].reset_index(drop=True)
            self.current_idx = 0

            print(f"Resuming: {len(processed_ids)} done, {len(remaining_ids)} remaining")

    def _create_widgets(self):
        """Create UI widgets."""
        def btn(desc, style, w='150px'):
            return widgets.Button(
                description=desc, button_style=style,
                layout=widgets.Layout(width=w, height='40px')
            )

        # Status buttons
        self.button_complete = btn('Complete', 'success')
        self.button_not_detected = btn('Not Detected', '')
        self.button_skip = btn('Skip', '')
        self.button_bad_weather = btn('Bad Weather', 'warning')
        self.button_malfunction = btn('Malfunction', 'danger')

        # Mic exclusion checkboxes
        self.checkbox_exclude_left = widgets.Checkbox(
            value=False, description='Exclude Left Mic',
            layout=widgets.Layout(width='200px')
        )
        self.checkbox_exclude_right = widgets.Checkbox(
            value=False, description='Exclude Right Mic',
            layout=widgets.Layout(width='200px')
        )

        # Spectrogram parameter controls
        self.slider_db = widgets.IntSlider(
            value=self.dynamic_range, min=20, max=100, step=5,
            description='dB Range:', layout=widgets.Layout(width='300px')
        )
        self.slider_gamma = widgets.FloatSlider(
            value=self.gamma, min=0.1, max=3.0, step=0.1,
            description='Gamma:', layout=widgets.Layout(width='300px')
        )
        self.button_refresh = btn('Refresh Spectrogram', 'info', '180px')

        # Audio controls
        self.button_play_left = btn('Play Left', 'info', '120px')
        self.button_play_right = btn('Play Right', 'info', '120px')
        self.button_play_both = btn('Play Both', 'info', '120px')
        self.button_stop = btn('Stop', '', '100px')

        # Navigation
        self.button_back = btn('Back', '', '100px')
        self.button_save = btn('Save Progress', 'primary')

        # Progress bar
        self.progress = widgets.IntProgress(
            value=len(self.results), min=0, max=self.total_detections,
            description='Progress:', bar_style='info',
            layout=widgets.Layout(width='600px')
        )

        self.status_text = widgets.HTML(value=self._get_status_html())
        self.box_info = widgets.HTML(value="")
        self.output = widgets.Output()

        # Wire up callbacks
        self.button_complete.on_click(lambda b: self._record('complete'))
        self.button_not_detected.on_click(lambda b: self._record('not_detected'))
        self.button_skip.on_click(lambda b: self._record('skip'))
        self.button_bad_weather.on_click(lambda b: self._record('bad_weather'))
        self.button_malfunction.on_click(lambda b: self._record('malfunction'))
        self.button_refresh.on_click(lambda b: self._refresh_spectrogram())
        self.button_play_left.on_click(
            lambda b: play_audio('left', self.current_audio_left,
                                 self.current_audio_right, self.current_sr))
        self.button_play_right.on_click(
            lambda b: play_audio('right', self.current_audio_left,
                                 self.current_audio_right, self.current_sr))
        self.button_play_both.on_click(
            lambda b: play_audio('both', self.current_audio_left,
                                 self.current_audio_right, self.current_sr))
        self.button_stop.on_click(lambda b: sd.stop())
        self.button_back.on_click(lambda b: self._go_back())
        self.button_save.on_click(lambda b: self._save_progress())

    def _get_status_html(self):
        """Generate status HTML."""
        completed = len(self.results)
        remaining = self.total_detections - completed

        stats = ""
        if self.results:
            results_df = pd.DataFrame(self.results)
            counts = results_df['status'].value_counts()
            parts = [f"<b>{s.replace('_', ' ').title()}:</b> {c}"
                     for s, c in counts.items()]
            stats = " | ".join(parts)

        return (
            f"<div style='font-size: 14px; padding: 10px; background-color: #f0f0f0; border-radius: 5px;'>"
            f"<b>Box Drawing Progress:</b> {completed} / {self.total_detections} "
            f"({remaining} remaining)<br>{stats}</div>"
        )

    def _on_box_select(self, eclick, erelease):
        """Callback when user draws a box on either spectrogram."""
        x1, y1 = eclick.xdata, eclick.ydata
        x2, y2 = erelease.xdata, erelease.ydata

        time_start = min(x1, x2)
        time_end = max(x1, x2)
        freq_min = min(y1, y2)
        freq_max = max(y1, y2)

        self.current_box = (time_start, time_end, freq_min, freq_max)
        self._update_box_display()

        self.box_info.value = (
            f"<b>Box:</b> Time: {time_start:.3f}s - {time_end:.3f}s | "
            f"Freq: {freq_min:.0f}Hz - {freq_max:.0f}Hz"
        )

    def _update_box_display(self):
        """Draw the bounding box on both spectrograms."""
        if self.current_box is None:
            return

        time_start, time_end, freq_min, freq_max = self.current_box
        width = time_end - time_start
        height = freq_max - freq_min

        if self.box_patch_left is not None:
            self.box_patch_left.remove()
        if self.box_patch_right is not None:
            self.box_patch_right.remove()

        self.box_patch_left = Rectangle(
            (time_start, freq_min), width, height,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        self.ax_left.add_patch(self.box_patch_left)

        self.box_patch_right = Rectangle(
            (time_start, freq_min), width, height,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        self.ax_right.add_patch(self.box_patch_right)

        self.fig.canvas.draw_idle()

    def _plot_dual_spectrograms(self, audio_left, audio_right, sr, row):
        """Display vertically stacked spectrograms for left and right channels."""
        self.dynamic_range = self.slider_db.value
        self.gamma = self.slider_gamma.value

        detection_duration = row['end_time'] - row['start_time']
        fig_width = detection_duration * 2
        fig_height = 5

        if self.fig is not None:
            plt.close(self.fig)

        self.fig, (self.ax_left, self.ax_right) = plt.subplots(
            2, 1, figsize=(fig_width, fig_height), sharex=True
        )
        self.fig.canvas.header_visible = False

        D_left = librosa.amplitude_to_db(
            np.abs(librosa.stft(audio_left, n_fft=512, hop_length=218)),
            ref=np.max
        )
        D_right = librosa.amplitude_to_db(
            np.abs(librosa.stft(audio_right, n_fft=512, hop_length=218)),
            ref=np.max
        )

        norm = PowerNorm(gamma=self.gamma, vmin=-self.dynamic_range, vmax=0)

        img_left = librosa.display.specshow(
            D_left, sr=sr, x_axis='time', y_axis='hz',
            hop_length=218, ax=self.ax_left, cmap='gray_r', norm=norm
        )
        self.ax_left.set_ylim(1000, 10000)
        self.ax_left.set_title('Left Microphone', fontsize=12)
        self.ax_left.set_xlabel('')

        img_right = librosa.display.specshow(
            D_right, sr=sr, x_axis='time', y_axis='hz',
            hop_length=218, ax=self.ax_right, cmap='gray_r', norm=norm
        )
        self.ax_right.set_ylim(1000, 10000)
        self.ax_right.set_title('Right Microphone', fontsize=12)

        self.fig.suptitle(
            f"Species: {row['species']} | Score: {row['confidence']:.3f} | "
            f"File: {Path(row['file_path']).name} | "
            f"Time: {row['start_time']:.1f}-{row['end_time']:.1f}s",
            fontsize=10
        )

        self.fig.colorbar(img_left, ax=self.ax_left, format='%+2.0f dB')
        self.fig.colorbar(img_right, ax=self.ax_right, format='%+2.0f dB')

        # Setup rectangle selectors on both spectrograms
        self.rect_selector_left = RectangleSelector(
            self.ax_left, self._on_box_select,
            useblit=True, button=[1],
            minspanx=0.01, minspany=50, spancoords='data',
            interactive=True,
            props=dict(facecolor='lime', edgecolor='lime', alpha=0.2, linewidth=2)
        )
        self.rect_selector_right = RectangleSelector(
            self.ax_right, self._on_box_select,
            useblit=True, button=[1],
            minspanx=0.01, minspany=50, spancoords='data',
            interactive=True,
            props=dict(facecolor='lime', edgecolor='lime', alpha=0.2, linewidth=2)
        )

        self.box_patch_left = None
        self.box_patch_right = None

        if self.current_box is not None:
            self._update_box_display()

        plt.tight_layout()
        plt.subplots_adjust(top=0.9)
        plt.show()

    def _show_detection(self):
        """Display current detection."""
        with self.output:
            clear_output(wait=True)

            if self.current_idx >= len(self.detections):
                self._extraction_complete()
                return

            row = self.detections.iloc[self.current_idx]

            # Reset box and checkboxes
            self.current_box = None
            self.checkbox_exclude_left.value = False
            self.checkbox_exclude_right.value = False
            self.box_info.value = ""

            audio_left, audio_right, sr = load_audio_stereo(
                row['file_path'], row['start_time'], row['end_time']
            )

            if audio_left is None:
                display(widgets.HTML(
                    f"<div style='color: red;'>Error loading audio: {row['file_path']}</div>"
                ))
                return

            self.current_audio_left = audio_left
            self.current_audio_right = audio_right
            self.current_sr = sr

            self._plot_dual_spectrograms(audio_left, audio_right, sr, row)

    def _refresh_spectrogram(self):
        """Refresh spectrogram with new dB/gamma settings."""
        if self.current_idx >= len(self.detections):
            return
        row = self.detections.iloc[self.current_idx]
        with self.output:
            clear_output(wait=True)
            if self.current_audio_left is not None:
                self._plot_dual_spectrograms(
                    self.current_audio_left, self.current_audio_right,
                    self.current_sr, row
                )

    def _record(self, status):
        """Record result and move to next detection."""
        if self.current_idx >= len(self.detections):
            return

        row = self.detections.iloc[self.current_idx].to_dict()
        row['status'] = status
        row['timestamp'] = datetime.now().isoformat()
        row['exclude_left_mic'] = self.checkbox_exclude_left.value
        row['exclude_right_mic'] = self.checkbox_exclude_right.value

        if self.current_box is not None and status == 'complete':
            row['box_start_time'] = self.current_box[0]
            row['box_end_time'] = self.current_box[1]
            row['box_freq_min'] = self.current_box[2]
            row['box_freq_max'] = self.current_box[3]
        else:
            row['box_start_time'] = None
            row['box_end_time'] = None
            row['box_freq_min'] = None
            row['box_freq_max'] = None

        self.results.append(row)
        self.current_idx += 1
        self.progress.value = len(self.results)
        self.status_text.value = self._get_status_html()

        if len(self.results) % 20 == 0:
            self._save_progress(silent=True)

        self._show_detection()

    def _go_back(self):
        """Go back to previous detection."""
        if len(self.results) == 0:
            with self.output:
                display(widgets.HTML(
                    "<div style='color: orange;'>Already at first detection</div>"
                ))
            return
        self.results.pop()
        self.current_idx = max(0, self.current_idx - 1)
        self.progress.value = len(self.results)
        self.status_text.value = self._get_status_html()
        self._show_detection()

    def _save_progress(self, silent=False):
        """Save progress to CSV."""
        if len(self.results) == 0:
            if not silent:
                with self.output:
                    display(widgets.HTML(
                        "<div style='color: orange;'>No results to save yet</div>"
                    ))
            return
        df = pd.DataFrame(self.results)
        df.to_csv(self.progress_file, index=False)
        if not silent:
            with self.output:
                display(widgets.HTML(
                    f"<div style='color: green;'>Saved {len(self.results)} results to {self.progress_file}</div>"
                ))

    def _extraction_complete(self):
        """Handle completion of all extractions."""
        df = pd.DataFrame(self.results)
        df.to_csv(self.results_file, index=False)

        counts = df['status'].value_counts()
        stats_lines = ''.join(
            f"<b>{s.replace('_', ' ').title()}:</b> {c}<br>"
            for s, c in counts.items()
        )

        display(widgets.HTML(f"""
        <div style='padding: 20px; background-color: #d4edda; border: 2px solid #28a745; border-radius: 10px;'>
            <h2 style='color: #155724;'>Box Drawing Complete!</h2>
            <p style='font-size: 16px;'>
                <b>Total processed:</b> {len(df)}<br>
                {stats_lines}
            </p>
            <p><b>Results saved to:</b> {self.results_file}</p>
        </div>
        """))

        print("\nResults by species:")
        print(df.groupby(['species', 'status']).size().unstack(fill_value=0))

    def display(self):
        """Show the detection box drawing interface."""
        status_buttons = widgets.HBox(
            [self.button_complete, self.button_not_detected, self.button_skip,
             self.button_bad_weather, self.button_malfunction],
            layout=widgets.Layout(justify_content='center')
        )
        mic_checkboxes = widgets.HBox(
            [self.checkbox_exclude_left, self.checkbox_exclude_right],
            layout=widgets.Layout(justify_content='center')
        )
        spec_controls = widgets.HBox(
            [self.slider_db, self.slider_gamma, self.button_refresh],
            layout=widgets.Layout(justify_content='center')
        )
        audio_controls = widgets.HBox(
            [self.button_play_left, self.button_play_right,
             self.button_play_both, self.button_stop],
            layout=widgets.Layout(justify_content='center')
        )
        nav_controls = widgets.HBox(
            [self.button_back, self.button_save],
            layout=widgets.Layout(justify_content='center')
        )

        display(self.status_text)
        display(self.progress)
        display(status_buttons)
        display(mic_checkboxes)
        display(self.box_info)
        display(spec_controls)
        display(audio_controls)
        display(nav_controls)
        display(self.output)

        self._show_detection()

In [ ]:
extractor = AmplitudeExtractionInterface(
    detections,
    progress_file=PROGRESS_FILE,
    results_file=RESULTS_FILE
)
extractor.display()

## 5. Review Results

After completing box drawing (or loading saved results), review the summary.

In [ ]:
# Load final results
results = pd.read_csv(RESULTS_FILE)

print(f"Total processed: {len(results)}")
print(f"\nOverall breakdown:")
print(results['status'].value_counts())

print(f"\nBy species:")
print(results.groupby(['species', 'status']).size().unstack(fill_value=0))

# Show detections with boxes
complete = results[results['status'] == 'complete']
print(f"\nDetections with bounding boxes: {len(complete)}")
if len(complete) > 0:
    print(complete[['species', 'box_start_time', 'box_end_time',
                     'box_freq_min', 'box_freq_max']].head(10))